In [115]:
import calendar
import os
import pandas as pd
from python.save_csv import save_df_to_csv
from datetime import timedelta, datetime

In [116]:
# temp read from temp files

DATA_DIR = r'K:/DOP/OED/METHOD&TOOLS/3 - PROJECTS/2 - ON GOING/PYTHON/Scripts/OEE2/python/data'

def read_csv(filename):
    path = os.path.join(DATA_DIR, filename)
    return pd.read_csv(path)

df_circ_data = read_csv('df_circ_data.csv')
df_meters = read_csv('df_meters.csv')
df_operations = read_csv('df_operations.csv')
df_osi_operations = read_csv('df_osi_operations.csv')
df_rigs = read_csv('df_rigs.csv')

In [117]:
df_osi_operations.describe(include='all')

,equipment,event,starttime,endtime
count,585938,585938,585938,585930
unique,88,119,51334,51367
top,H757214 URAL 4320 URB 3A3 №4,WT_PPR,2024-07-19 09:00:00,2024-05-31 08:00:00
freq,17726,152847,60,60


In [118]:
df_meters.describe(include='all')

,HOLEID,DrillCompany,DrillRig,HolePurpose,HoleStatus,ENDDATE,DEPTH
count,7417,7417,7417,7417,7417,7417,7363.000000
unique,7417,6,83,4,3,1670,NaN
top,TNU37_03_16_A_,BurGeoProekt,ZIF1200_29-BGP,INJECT,ACCEPTED,2025-01-01,NaN
freq,1,3336,212,5203,6791,58,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,426.577434
std,NaN,NaN,NaN,NaN,NaN,NaN,86.246690
min,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
25%,NaN,NaN,NaN,NaN,NaN,NaN,394.500000
50%,NaN,NaN,NaN,NaN,NaN,NaN,435.000000
75%,NaN,NaN,NaN,NaN,NaN,NaN,500.000000


In [119]:
# Преобразуем Starttime и Endtime в datetime
if df_osi_operations is not None:
    df_osi_operations['starttime'] = pd.to_datetime(df_osi_operations['starttime'], errors='coerce')
    df_osi_operations['endtime'] = pd.to_datetime(df_osi_operations['endtime'], errors='coerce')

In [120]:
df_meters['ENDDATE'] = pd.to_datetime(df_meters['ENDDATE'])

In [121]:
# Функция для преобразования имён колонок:
def clean_column_names(columns):
    return [col.replace('_', ' ').title() for col in columns]

# Очищаем и переименовываем колонки сразу после чтения
if df_circ_data is not None:
    df_circ_data.columns = clean_column_names(df_circ_data.columns)

if df_meters is not None:
    df_meters.columns = clean_column_names(df_meters.columns)
    
if df_operations is not None:   
    df_operations.columns = clean_column_names(df_operations.columns)
    
if df_osi_operations is not None:  
    df_osi_operations.columns = clean_column_names(df_osi_operations.columns)
    
if df_rigs is not None:
    df_rigs.columns = clean_column_names(df_rigs.columns)

In [122]:
def add_difference_column(df, col1, col2):
    """
    Возвращает Series: разница между двумя колонками с заменой NaN на 0.
    """
    return (df[col1].fillna(0) - df[col2].fillna(0)).clip(lower=0)


def add_ratio_column(df, numerator_col, denominator_col):
    """
    Возвращает Series: отношение numerator_col / denominator_col
    с защитой от деления на 0, заменой NaN на 0 и ограничением [0, 1].
    """
    num = df[numerator_col].fillna(0)
    denom = df[denominator_col].replace(0, pd.NA).fillna(pd.NA)
    ratio = num / denom
    return ratio.fillna(0).clip(0, 1)

In [123]:
df_osi_operations

,Equipment,Event,Starttime,Endtime
0,DBKAZ40 BL №1,RP_REPAIR,2021-08-20 11:00:00,2021-09-02 16:00:00
1,DBKAZ40 BL №1,CV_LONG_STOP,2020-04-27 07:00:00,2020-08-15 07:00:00
2,DBKAZ40 BL №1,WT_PPR,2020-02-18 07:00:00,2020-02-18 08:00:00
3,DBKAZ40 BL №1,WT_LUNCH,2020-02-18 11:00:00,2020-02-18 12:00:00
4,DBKAZ40 BL №1,WT_PPR,2020-02-18 19:00:00,2020-02-18 20:00:00
...,...,...,...,...
585933,ЯЯЯЯ-ФЕЙК,DR_CORE_DRILLING,2024-09-29 20:00:00,2024-09-29 22:30:00
585934,ЯЯЯЯ-ФЕЙК,DR_REAMING,2024-09-30 10:30:00,2024-09-30 12:30:00
585935,ЯЯЯЯ-ФЕЙК,DR_PUMP_REPLACEMENT,2024-09-30 13:00:00,2024-09-30 14:30:00
585936,ЯЯЯЯ-ФЕЙК,DR_WELL_FLUSHING,2024-10-01 11:30:00,2024-10-01 14:00:00


In [124]:
# Работа с df_operations
df_operations = df_operations[['Event', 'Category']] # pyright: ignore[reportOptionalSubscript]
df_operations = df_operations[df_operations['Category'].notna()].reset_index(drop=True)

In [125]:
if df_rigs is not None:
    df_rigs['Rig-Osidem'] = df_rigs['Rig-Osidem'].str.strip().str.lower()

In [126]:
if df_osi_operations is not None:
    df_osi_operations['Equipment'] = df_osi_operations['Equipment'].str.strip().str.lower()

In [127]:
rows_out = []

for row in df_osi_operations.itertuples(index=False):
    start = row.Starttime
    end = row.Endtime
    total_seconds = (end - start).total_seconds()

    if start.year == end.year and start.month == end.month:
        # Месяц не меняется — ничего не делим
        rows_out.append({
            **row._asdict(),
            'Year-Month': start.strftime('%Y-%m'),
            'Duration': total_seconds / 3600
        })
    else:
        # Делаем разбивку
        current_start = start
        while current_start < end:
            next_month = (current_start.replace(day=1) + timedelta(days=32)).replace(day=1)
            segment_end = min(end, next_month)
            segment_seconds = (segment_end - current_start).total_seconds()

            rows_out.append({
                **row._asdict(),
                'Starttime': current_start,
                'Endtime': segment_end,
                'Year-Month': current_start.strftime('%Y-%m'),
                'Duration': round((segment_seconds / total_seconds) * ((end - start).total_seconds() / 3600), 2)
            })

            current_start = segment_end
            
            


df_osi_operations = pd.DataFrame(rows_out)




In [128]:
df_osi_operations

,Equipment,Event,Starttime,Endtime,Year-Month,Duration
0,dbkaz40 bl №1,RP_REPAIR,2021-08-20 11:00:00,2021-09-01 11:00:00,2021-08,288.0
1,dbkaz40 bl №1,RP_REPAIR,2021-09-01 11:00:00,2021-09-02 16:00:00,2021-09,29.0
2,dbkaz40 bl №1,CV_LONG_STOP,2020-04-27 07:00:00,2020-05-01 07:00:00,2020-04,96.0
3,dbkaz40 bl №1,CV_LONG_STOP,2020-05-01 07:00:00,2020-06-01 07:00:00,2020-05,744.0
4,dbkaz40 bl №1,CV_LONG_STOP,2020-06-01 07:00:00,2020-07-01 07:00:00,2020-06,720.0
...,...,...,...,...,...,...
586216,яяяя-фейк,DR_CORE_DRILLING,2024-09-29 20:00:00,2024-09-29 22:30:00,2024-09,2.5
586217,яяяя-фейк,DR_REAMING,2024-09-30 10:30:00,2024-09-30 12:30:00,2024-09,2.0
586218,яяяя-фейк,DR_PUMP_REPLACEMENT,2024-09-30 13:00:00,2024-09-30 14:30:00,2024-09,1.5
586219,яяяя-фейк,DR_WELL_FLUSHING,2024-10-01 11:30:00,2024-10-01 14:00:00,2024-10,2.5


In [129]:
df_osi_operations[df_osi_operations['Event'] == 'WT_LUNCH']

,Equipment,Event,Starttime,Endtime,Year-Month,Duration
8,dbkaz40 bl №1,WT_LUNCH,2020-02-18 11:00:00,2020-02-18 12:00:00,2020-02,1.0
13,dbkaz40 bl №1,WT_LUNCH,2020-02-18 23:00:00,2020-02-19 00:00:00,2020-02,1.0
18,dbkaz40 bl №1,WT_LUNCH,2020-02-19 23:00:00,2020-02-20 00:00:00,2020-02,1.0
20,dbkaz40 bl №1,WT_LUNCH,2020-02-20 12:00:00,2020-02-20 13:00:00,2020-02,1.0
22,dbkaz40 bl №1,WT_LUNCH,2020-02-21 00:00:00,2020-02-21 01:00:00,2020-02,1.0
...,...,...,...,...,...,...
582389,пбу змо-1500пс №6,WT_LUNCH,2025-09-03 00:00:00,2025-09-03 01:00:00,2025-09,1.0
582391,пбу змо-1500пс №6,WT_LUNCH,2025-09-03 12:00:00,2025-09-03 13:00:00,2025-09,1.0
582393,пбу змо-1500пс №6,WT_LUNCH,2025-09-04 00:00:00,2025-09-04 01:00:00,2025-09,1.0
582825,пбу змо-1500пс №6,WT_LUNCH,2023-11-05 11:00:00,2023-11-05 13:00:00,2023-11,2.0


In [130]:
# Слияние
if df_osi_operations is not None and df_rigs is not None:
    df_merged = df_osi_operations.merge( # pyright: ignore[reportOptionalMemberAccess]
        df_rigs,
        left_on='Equipment',
        right_on='Rig-Osidem',
        how='left'
    )

In [131]:
df_merged = df_merged[df_merged['Rig-Acquire'].notna()].reset_index(drop=True)
df_temp = df_merged.drop(columns=['Equipment', 'Rig-Osidem'])

In [132]:
# Очистка колонок event
df_temp['Event'] = df_temp['Event'].str.strip().str.lower()
df_operations['Event'] = df_operations['Event'].str.strip().str.lower()

In [133]:
# Слияние с категориями
df_operations_total = df_temp.merge(df_operations, on='Event', how='left')
df_operations_total['Category'] = df_operations_total['Category'].fillna('Standard Work')

In [134]:
# Добавляем year и month
df_operations_total['Year-Month'] = df_operations_total['Starttime'].dt.strftime('%Y-%m')

In [135]:
df_operations_total

,Event,Starttime,Endtime,Year-Month,Duration,Drillcompany,Rig-Acquire,Tipe Of Circulation,Category
0,rp_repair,2021-08-20 11:00:00,2021-09-01 11:00:00,2021-08,288.0,KATCO,DBKAZ40-1,RC,Unplanned_downtime_losses
1,rp_repair,2021-09-01 11:00:00,2021-09-02 16:00:00,2021-09,29.0,KATCO,DBKAZ40-1,RC,Unplanned_downtime_losses
2,cv_long_stop,2020-04-27 07:00:00,2020-05-01 07:00:00,2020-04,96.0,KATCO,DBKAZ40-1,RC,Standard Work
3,cv_long_stop,2020-05-01 07:00:00,2020-06-01 07:00:00,2020-05,744.0,KATCO,DBKAZ40-1,RC,Standard Work
4,cv_long_stop,2020-06-01 07:00:00,2020-07-01 07:00:00,2020-06,720.0,KATCO,DBKAZ40-1,RC,Standard Work
...,...,...,...,...,...,...,...,...,...
545679,dv_development,2025-09-03 09:00:00,2025-09-03 12:00:00,2025-09,3.0,KATCO,ZMO1500-6-KAT,Direct,Standard Work
545680,dv_development,2025-09-03 13:00:00,2025-09-03 20:00:00,2025-09,7.0,KATCO,ZMO1500-6-KAT,Direct,Standard Work
545681,dv_development,2025-09-03 21:00:00,2025-09-04 00:00:00,2025-09,3.0,KATCO,ZMO1500-6-KAT,Direct,Standard Work
545682,dv_development,2025-09-04 01:00:00,2025-09-04 08:00:00,2025-09,7.0,KATCO,ZMO1500-6-KAT,Direct,Standard Work


In [136]:
df_operations_total = df_operations_total[df_operations_total['Event'] != 'wt_lunch']

# Получаем уникальные Year-Month и Rig-Acquire
unique_combinations = (
    df_operations_total[
        ['Year-Month', 'Rig-Acquire', 'Drillcompany', 'Tipe Of Circulation']
    ]
    .drop_duplicates()
)




# Список строк с ланчами
lunch_rows = []

for _, row in unique_combinations.iterrows():
    ym = row['Year-Month']
    rig = row['Rig-Acquire']
    company = row['Drillcompany']
    circulation = row['Tipe Of Circulation']
    
    year, month = map(int, ym.split('-'))
    days_in_month = calendar.monthrange(year, month)[1]
    
    duration = days_in_month * 2  # по 2 часа в день
    start_time = pd.Timestamp(f"{ym}-01 13:00:00")
    end_time = start_time + pd.Timedelta(hours=duration)

    lunch_rows.append({
        'Event': 'wt_lunch',
        'Starttime': start_time,
        'Endtime': end_time,
        'Duration': duration,
        'Year-Month': ym,
        'Drillcompany': company,
        'Rig-Acquire': rig,
        'Tipe Of Circulation': circulation,
        'Category': 'Planned_downtime',
    })

# Создаём DataFrame и добавляем к основному
df_lunch = pd.DataFrame(lunch_rows)
df_operations_total = pd.concat([df_operations_total, df_lunch], ignore_index=True)

# Сброс индекса
df_operations_total = df_operations_total.reset_index(drop=True)




In [137]:
df_operations_total

,Event,Starttime,Endtime,Year-Month,Duration,Drillcompany,Rig-Acquire,Tipe Of Circulation,Category
0,rp_repair,2021-08-20 11:00:00,2021-09-01 11:00:00,2021-08,288.0,KATCO,DBKAZ40-1,RC,Unplanned_downtime_losses
1,rp_repair,2021-09-01 11:00:00,2021-09-02 16:00:00,2021-09,29.0,KATCO,DBKAZ40-1,RC,Unplanned_downtime_losses
2,cv_long_stop,2020-04-27 07:00:00,2020-05-01 07:00:00,2020-04,96.0,KATCO,DBKAZ40-1,RC,Standard Work
3,cv_long_stop,2020-05-01 07:00:00,2020-06-01 07:00:00,2020-05,744.0,KATCO,DBKAZ40-1,RC,Standard Work
4,cv_long_stop,2020-06-01 07:00:00,2020-07-01 07:00:00,2020-06,720.0,KATCO,DBKAZ40-1,RC,Standard Work
...,...,...,...,...,...,...,...,...,...
531966,wt_lunch,2025-05-01 13:00:00,2025-05-04 03:00:00,2025-05,62.0,KATCO,ZMO1500-6-KAT,Direct,Planned_downtime
531967,wt_lunch,2025-06-01 13:00:00,2025-06-04 01:00:00,2025-06,60.0,KATCO,ZMO1500-6-KAT,Direct,Planned_downtime
531968,wt_lunch,2025-07-01 13:00:00,2025-07-04 03:00:00,2025-07,62.0,KATCO,ZMO1500-6-KAT,Direct,Planned_downtime
531969,wt_lunch,2025-08-01 13:00:00,2025-08-04 03:00:00,2025-08,62.0,KATCO,ZMO1500-6-KAT,Direct,Planned_downtime


In [138]:
# Сводная таблица pivot
pivot_df = df_operations_total.pivot_table(
    index=['Drillcompany', 'Rig-Acquire', 'Year-Month', 'Tipe Of Circulation'],
    columns='Category',
    values='Duration',
    aggfunc='sum',
    fill_value=0
).reset_index()


In [139]:
pivot_df

Category,Drillcompany,Rig-Acquire,Year-Month,Tipe Of Circulation,Planned_downtime,Standard Work,Unplanned_downtime_losses
0,BurGeoProekt,ZIF1200_01-BGP,2020-02,Direct,58.0,8.0,276.0
1,BurGeoProekt,ZIF1200_01-BGP,2020-03,Direct,109.0,423.0,278.0
2,BurGeoProekt,ZIF1200_01-BGP,2020-04,Direct,77.0,675.0,28.0
3,BurGeoProekt,ZIF1200_01-BGP,2020-05,Direct,62.0,744.0,0.0
4,BurGeoProekt,ZIF1200_01-BGP,2020-06,Direct,60.0,720.0,0.0
...,...,...,...,...,...,...,...
3076,TechnoService-Eng,ZIF1200_16-TSE,2023-03,Direct,151.0,520.0,90.0
3077,TechnoService-Eng,ZIF1200_16-TSE,2023-04,Direct,136.0,462.0,106.0
3078,TechnoService-Eng,ZIF1200_16-TSE,2023-05,Direct,138.0,488.0,119.0
3079,TechnoService-Eng,ZIF1200_16-TSE,2023-06,Direct,137.0,405.0,133.0


In [140]:
pivot_df = pivot_df.sort_values(
    by=['Drillcompany', 'Rig-Acquire', 'Year-Month'],
    ascending=[True, True, True]
).reset_index(drop=True)

# Функция для подсчёта часов в месяце
def hours_in_month(row):
    year_month_str = row['Year-Month']  # предполагается, что у вас есть колонка с таким именем
    year, month = map(int, year_month_str.split('-'))
    days = calendar.monthrange(year, month)[1]
    return days * 24


pivot_df['H In Month'] = pivot_df.apply(hours_in_month, axis=1)


# Преобразование ENDDATE в datetime
if df_meters is not None and df_rigs is not None:
    df_meters['Enddate'] = pd.to_datetime(df_meters['Enddate'], errors='coerce')
    df_meters.dropna(subset=['Enddate'], inplace=True)
    df_meters['Year-Month'] = df_meters['Enddate'].dt.strftime('%Y-%m')




In [141]:
if df_meters is not None:
    pivot2 = pd.pivot_table(
        df_meters,
        index=['Year-Month', 'Drillrig'],
        columns='Holestatus',
        values='Holeid',
        aggfunc='count',
        fill_value=0
    )


if df_meters is not None:
    depth_sum = df_meters.groupby(['Year-Month', 'Drillrig'])['Depth'].sum()
    
    
pivot2['Depth'] = depth_sum
meters_pivot = pivot2.reset_index()

# Приведение к верхнему регистру и очистка пробелов
pivot_df['Rig-Acquire'] = pivot_df['Rig-Acquire'].str.strip().str.upper()
meters_pivot['Drillrig'] = meters_pivot['Drillrig'].str.strip().str.upper()

total_merged_df = pd.merge(
    pivot_df,
    meters_pivot,
    left_on=['Rig-Acquire', 'Year-Month'],
    right_on=['Drillrig', 'Year-Month'],
    how='left'
)


In [142]:
total_merged_df.drop(columns='Rig-Acquire', inplace=True)


# Расчёты Planned Production Time и Planned Factor с ограничениями
total_merged_df['Planned Production Time'] = add_difference_column(total_merged_df, 'H In Month', 'Planned_downtime')


total_merged_df['Planned Factor'] = add_ratio_column(
    total_merged_df, 'Planned Production Time', 'H In Month'
)

# Gross Operating Time (GOT) и Availability
total_merged_df['Gross Operating Time'] = add_difference_column(
    total_merged_df, 'Planned Production Time', 'Unplanned_downtime_losses'
)


In [143]:
total_merged_df['Availability'] = add_ratio_column(
    total_merged_df, 'Gross Operating Time', 'Planned Production Time'
)

# Назначаем коэффициенты по типу циркуляции
if df_circ_data is not None and 'Circ' in df_circ_data.columns and 'Standard Avarage Drilling, M/H' in df_circ_data.columns:
    circ_avg_drilling = df_circ_data.set_index('Circ')['Standard Avarage Drilling, M/H'].to_dict()
else:
    circ_avg_drilling = {}
    
    
if df_circ_data is not None and 'Circ' in df_circ_data.columns and 'Time To Well Drill, H' in df_circ_data.columns:
    time_to_well_drill = df_circ_data.set_index('Circ')['Time To Well Drill, H'].to_dict()
else:
    time_to_well_drill = {}


In [144]:
# Записываем два столбца в df
total_merged_df['Circulation Coeff'] = total_merged_df['Tipe Of Circulation'].map(circ_avg_drilling)
total_merged_df['Well Drill Coef'] = total_merged_df['Tipe Of Circulation'].map(time_to_well_drill)


# Потенциальная глубина бурения
total_merged_df['Potential Depth'] = (
    total_merged_df['Gross Operating Time'] * total_merged_df['Circulation Coeff']
)


total_merged_df['Net Operating Time'] = total_merged_df['Depth'].div(
    total_merged_df['Circulation Coeff']
).fillna(0)


total_merged_df['Speed Losses'] = add_difference_column(
    total_merged_df, 'Gross Operating Time', 'Net Operating Time'
)



In [145]:
total_merged_df['Performance'] = add_ratio_column(total_merged_df, 'Net Operating Time', 'Gross Operating Time')

total_merged_df['Quality Losses'] =  total_merged_df.get('Well Drill Coef', 0) * total_merged_df.get('LIQUID', 0)

# Valuable Operating Time
total_merged_df['Valuable Operating Time'] = add_difference_column(
    total_merged_df, 'Net Operating Time', 'Quality Losses'
    )



total_merged_df['Quality'] = add_ratio_column(
    total_merged_df, 'Valuable Operating Time', 'Net Operating Time'
    )


total_merged_df['OEE'] = (
    total_merged_df['Availability'] *
    total_merged_df['Performance'] *
    total_merged_df['Quality']
)


C:\Users\ykarabekov\AppData\Local\Temp\ipykernel_21948\516393728.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return ratio.fillna(0).clip(0, 1)
C:\Users\ykarabekov\AppData\Local\Temp\ipykernel_21948\516393728.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return ratio.fillna(0).clip(0, 1)


In [146]:
total_merged_df

,Drillcompany,Year-Month,Tipe Of Circulation,Planned_downtime,Standard Work,Unplanned_downtime_losses,H In Month,Drillrig,ACCEPTED,LIQUID,...,Circulation Coeff,Well Drill Coef,Potential Depth,Net Operating Time,Speed Losses,Performance,Quality Losses,Valuable Operating Time,Quality,OEE
0,BurGeoProekt,2020-02,Direct,58.0,8.0,276.0,696,ZIF1200_01-BGP,0.0,1.0,...,2.7,210,977.4,29.629630,332.370370,0.081850,210.0,0.000000,0.000000,0.000000
1,BurGeoProekt,2020-03,Direct,109.0,423.0,278.0,744,ZIF1200_01-BGP,2.0,0.0,...,2.7,210,963.9,258.518519,98.481481,0.724142,0.0,258.518519,1.000000,0.407116
2,BurGeoProekt,2020-04,Direct,77.0,675.0,28.0,720,ZIF1200_01-BGP,2.0,0.0,...,2.7,210,1660.5,258.148148,356.851852,0.419753,0.0,258.148148,1.000000,0.401475
3,BurGeoProekt,2020-05,Direct,62.0,744.0,0.0,744,NaN,NaN,NaN,...,2.7,210,1841.4,0.000000,682.000000,0.000000,NaN,0.000000,0.000000,0.000000
4,BurGeoProekt,2020-06,Direct,60.0,720.0,0.0,720,NaN,NaN,NaN,...,2.7,210,1782.0,0.000000,660.000000,0.000000,NaN,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3076,TechnoService-Eng,2023-03,Direct,151.0,520.0,90.0,744,ZIF1200_16-TSE,2.0,0.0,...,2.7,210,1358.1,217.407407,285.592593,0.432221,0.0,217.407407,1.000000,0.366623
3077,TechnoService-Eng,2023-04,Direct,136.0,462.0,106.0,720,ZIF1200_16-TSE,4.0,0.0,...,2.7,210,1290.6,419.259259,58.740741,0.877111,0.0,419.259259,1.000000,0.717910
3078,TechnoService-Eng,2023-05,Direct,138.0,488.0,119.0,744,ZIF1200_16-TSE,2.0,0.0,...,2.7,210,1314.9,209.259259,277.740741,0.429690,0.0,209.259259,1.000000,0.345312
3079,TechnoService-Eng,2023-06,Direct,137.0,405.0,133.0,720,ZIF1200_16-TSE,3.0,0.0,...,2.7,210,1215.0,386.296296,63.703704,0.858436,0.0,386.296296,1.000000,0.662601


In [147]:
total_merged_df['TRS'] = total_merged_df['OEE'] * total_merged_df['Planned Factor']


total_merged_df.columns = clean_column_names(total_merged_df.columns)


total_merged_df = total_merged_df.rename(columns={'Drillrig': 'Rig'})


df_losses = pd.melt(
    total_merged_df,
    id_vars=[
        'Drillcompany',
        'Rig',
        'Year-Month',
        'Tipe Of Circulation'
    ],
    value_vars=[
        'Planned Downtime',
        'Unplanned Downtime Losses',
        'Speed Losses',
        'Quality Losses'
    ],
    var_name='Loss Type',
    value_name='Loss Value'
)


In [148]:
df_productivity = pd.melt(
    total_merged_df,
    id_vars=[
        'Drillcompany',
        'Rig',
        'Year-Month',
        'Tipe Of Circulation'
    ],
    value_vars=[
        'Planned Factor',
        'Availability',
        'Performance',
        'Quality',
        'Oee',
        'Trs'
    ],
    var_name='Productivity Type',
    value_name='Productivity Value'
)


df_operations_total['Category'] = df_operations_total['Category'].replace({
    'Planned_downtime': 'Planned Downtime',
    'Unplanned_downtime_losses': 'Unplanned Downtime Losses'
})


df_operations_total = df_operations_total.rename(columns={'Rig-Acquire': 'Rig'})


df_operations_total['Event Category'] = df_operations_total['Event'].str.split('_').str[0].str.upper()
df_operations_total['Event'] = df_operations_total['Event'].str.split('_', n=1).str[1].str.capitalize()


df_events_duration = df_operations_total[
    [
        'Year-Month',
        "Category",
        'Drillcompany',
        'Rig',
        'Tipe Of Circulation',
        "Event Category",
        "Event",
        "Duration"
    ]
].sort_values(
    by=['Year-Month', 'Rig', 'Category', 'Duration'],
    ascending=[True, True, True, False],
)


In [149]:
if df_meters is not None:
    df_meters.dropna(subset=['Drillcompany', 'Drillrig'], inplace=True)
    
    df_meters.columns = [
    'Hole ID',
    'Drilling Company',
    'Rig',
    'Purpose',
    'Status',
    'End_Date',
    'Depth_m',
    'Year-Month'
]

    df_meters = df_meters.drop(columns=['End_Date'])


In [150]:
unique_dates_series = pd.Series(
    sorted(df_productivity['Year-Month'].dropna().unique()),
    name='Year-Month'
)

unique_companies_series = pd.Series(
    sorted(df_productivity['Drillcompany'].dropna().unique()),
    name='Companies'
)

unique_rigs_series = pd.Series(
    sorted(df_productivity['Rig'].dropna().unique()),
    name='Rig'
)

unique_circ_series = pd.Series(
    sorted(df_productivity['Tipe Of Circulation'].dropna().unique()),
    name='Circ Type'
)

unique_product_type_series = pd.Series(
    sorted(df_productivity['Productivity Type'].dropna().unique()),
    name='Productivity Type'
)

unique_losses_series = pd.Series(
    sorted(df_losses['Loss Type'].dropna().unique()),
    name='Loss Type'
)


In [151]:
folder = r'k:/DOP/OED/METHOD&TOOLS/3 - PROJECTS/2 - ON GOING/2 - OE/2502 DIGITAL PROJECTS YEVGENIY/TRS/extr_csv_files/'

save_df_to_csv(df_meters, 'df_meters.csv', folder)
save_df_to_csv(df_events_duration, 'df_events_duration.csv', folder)
save_df_to_csv(df_productivity, 'df_productivity.csv', folder)
save_df_to_csv(df_losses, 'df_losses.csv', folder)
save_df_to_csv(unique_dates_series, 'df_dates.csv', folder)

Файл сохранён: k:/DOP/OED/METHOD&TOOLS/3 - PROJECTS/2 - ON GOING/2 - OE/2502 DIGITAL PROJECTS YEVGENIY/TRS/extr_csv_files/df_meters.csv
Файл сохранён: k:/DOP/OED/METHOD&TOOLS/3 - PROJECTS/2 - ON GOING/2 - OE/2502 DIGITAL PROJECTS YEVGENIY/TRS/extr_csv_files/df_events_duration.csv
Файл сохранён: k:/DOP/OED/METHOD&TOOLS/3 - PROJECTS/2 - ON GOING/2 - OE/2502 DIGITAL PROJECTS YEVGENIY/TRS/extr_csv_files/df_productivity.csv
Файл сохранён: k:/DOP/OED/METHOD&TOOLS/3 - PROJECTS/2 - ON GOING/2 - OE/2502 DIGITAL PROJECTS YEVGENIY/TRS/extr_csv_files/df_losses.csv
Файл сохранён: k:/DOP/OED/METHOD&TOOLS/3 - PROJECTS/2 - ON GOING/2 - OE/2502 DIGITAL PROJECTS YEVGENIY/TRS/extr_csv_files/df_dates.csv


In [152]:
df_meters

,Hole ID,Drilling Company,Rig,Purpose,Status,Depth_m,Year-Month
0,MSK39_08_09_A4,BurGeoProekt,PRAKLA_01-BGP,PRODUCT,ACCEPTED,508.0,2020-04
1,MSK03_06_02_A_,BurGeoProekt,PRAKLA_03-BGP,PRODUCT,LIQUID,475.0,2020-04
2,MSK03_04_05_A_,BurGeoProekt,PRAKLA_04-BGP,PRODUCT,LIQUID,503.0,2020-09
3,MSK03_04_09_A_,BurGeoProekt,PRAKLA_04-BGP,PRODUCT,LIQUID,506.0,2020-09
4,MSK04_05_04_A_,BurGeoProekt,ZIF1200_29-BGP,INJECT,ACCEPTED,498.0,2020-09
...,...,...,...,...,...,...,...
7412,TNU37_01_13_A_,TechnoService-Eng,ZIF1200_04-TSE,INJECT,ACCEPTED,303.0,2025-08
7413,TNU37_03_13_A_,TechnoService-Eng,ZIF1200_14-TSE,INJECT,ACCEPTED,307.0,2025-08
7414,TNU37_01_14_A_,TechnoService-Eng,ZIF1200_03-TSE,INJECT,ACCEPTED,303.0,2025-08
7415,TNU37_03_14_A_,TechnoService-Eng,ZIF1200_04-TSE,INJECT,ACCEPTED,306.0,2025-08
